# Searching with Cut-based distance

## Import computed cuts

In [ ]:
dist_method = 0  # 0 for ED, 1 for DTW, 15 for DTW (self-defined)
# function_used: psdtw_prime_vanilla, psdtw_prime_parallel
function_used = psdtw_prime_vanilla

In [ ]:
data = np.load(
    f"../outputs/{dataset_name}_P{P}_l{l:.2f}_dist_method{dist_method}_{function_used.__name__}.npz",
    # f"../outputs_old/{dataset_name}_P{P}_l{l:.2f}_dist_method{dist_method}_{function_used.__name__}.npz",
    allow_pickle=True,
)

all_distances = np.ascontiguousarray(data["all_distances"], dtype=np.float64)
all_count_dist_calls = np.ascontiguousarray(
    data["all_count_dist_calls"], dtype=np.float64
)
all_cuts = np.ascontiguousarray(data["all_cuts"], dtype=np.float64)

In [ ]:
print(
    f"{data["precision_at_1"]:.2f}",
    f"{data["precision_at_3"]:.2f}",
    end=" & ",
    # f"{data["precision_at_5"]:.2f}",
    # f"{data["precision_at_7"]:.2f}",
)
# print(
#     f"{precision_at_1 / len(query_set):.2f} & {precision_at_3 / len(query_set):.2f}",
#     end=" & ",
# )
print()
print("Elapsed time:", data["elapsed_time"])

total_count_dist_calls = 0
for r in all_count_dist_calls:
    total_count_dist_calls += np.sum(r)
print("Total distance measure calls: " + str(total_count_dist_calls))

0.94 1.00 & 
Elapsed time: 30.447548389434814
Total distance measure calls: 60658171.0


## Compute Cut-based distances

In [ ]:
# 0: aeon_squared_distance, 1: aeon_dtw_distance, 2: aeon_adtw_distance, 3: aeon_ddtw_distance, 4: aeon_erp_distance, 5: aeon_edr_distance
# 6: aeon_lcss_distance, 7: aeon_manhattan_distance, 8: aeon_minkowski_distance, 9: aeon_msm_distance, 10: aeon_sbd_distance
# 11: aeon_shape_dtw_distance, 12: aeon_twe_distance, 13: aeon_wddtw_distance, 14: aeon_wdtw_distance
# for i in range(0, 15):
for i in [1, 2, 3, 11, 13, 14]:
    # print("dist_method: " + str(i))
    dist_method = i
    precision_at_1, precision_at_3, precision_at_5, precision_at_7 = 0, 0, 0, 0
    for i in range(0, len(query_set)):
        distances = np.array(
            [
                cut_based_distance(
                    query_set[i],
                    target_set[j],
                    0.1,
                    l,
                    P,
                    dist_method=dist_method,
                    # cuts=all_cuts[i][j],
                    cuts=all_cuts[i][j],
                )
                for j in range(0, len(target_set))
            ]
        )
        precision_at_1 += precision_at_k(distances, i, 1)
        precision_at_3 += precision_at_k(distances, i, 3)
        # precision_at_5 += precision_at_k(distances, i, 5)
        # precision_at_7 += precision_at_k(distances, i, 7)
    print(
        f"{precision_at_1 / len(query_set):.2f} & {precision_at_3 / len(query_set):.2f}",
        end=" & ",
    )
    # print(
    #     f"{precision_at_1 / len(query_set):.2f}",
    #     f"{precision_at_3 / len(query_set):.2f}",
    # )

0.82 & 0.96 & 0.94 & 1.00 & 0.74 & 0.90 & 0.92 & 0.98 & 0.76 & 0.90 & 0.82 & 0.96 & 

# Testing with PSD methods

In [ ]:
instance_idx = 0
Q = query_set[instance_idx]
C = target_set[instance_idx]

r =0.1
l=l
P=P
dist_method=15

In [ ]:
# Warmup for numba
psdtw_prime_vanilla(Q, C, r=r, l=l, P=P, dist_method=dist_method)

(1.0202125612107882,
 25110,
 array([[  0,  49,   0,  51],
        [ 49, 103,  51, 105],
        [103, 150, 105, 150]]))

In [ ]:
start = time.time()
psdtw_prime_vanilla(Q, C, r=r, l=l, P=P, dist_method=dist_method)
end = time.time()
print(f"Elapsed time: {end - start} seconds")


Elapsed time: 0.05864238739013672 seconds


In [ ]:
# Warmup for numba
psdtw_prime_parallel(Q, C, r=r, l=l, P=P, dist_method=dist_method)

(1.0202125612107882,
 25110,
 array([[  0,  49,   0,  51],
        [ 49, 103,  51, 105],
        [103, 150, 105, 150]]))

In [ ]:
start = time.time()
psdtw_prime_parallel(Q, C, r=r, l=l, P=P, dist_method=dist_method)
end = time.time()
print(f"Elapsed time: {end - start} seconds")

Elapsed time: 0.025814056396484375 seconds


In [ ]:
# Warmup for numba
psdtw_prime_parallel_bsf(Q, C, r=r, l=l, P=P, dist_method=dist_method)

(1.0202125612107882,
 25110,
 array([[  0,  49,   0,  51],
        [ 49, 103,  51, 105],
        [103, 150, 105, 150]]))

In [ ]:
start = time.time()
psdtw_prime_parallel_bsf(Q, C, r=r, l=l, P=P, dist_method=dist_method)
end = time.time()
print(f"Elapsed time: {end - start} seconds")

Elapsed time: 0.0095367431640625 seconds


In [ ]:
# Warmup for numba
psdtw_prime_parallel_bsf_lb(Q, C, r=r, l=l, P=P, dist_method=dist_method)

(1.0202125612107882,
 20914,
 array([[  0,  49,   0,  51],
        [ 49, 103,  51, 105],
        [103, 150, 105, 150]]))

In [ ]:
start = time.time()
psdtw_prime_parallel_bsf_lb(Q, C, r=r, l=l, P=P, dist_method=dist_method)
end = time.time()
print(f"Elapsed time: {end - start} seconds")

Elapsed time: 0.014683246612548828 seconds


In [ ]:
# Warmup for numba
psdtw_prime_parallel_bsf_lb2(Q, C, r=r, l=l, P=P, dist_method=dist_method)

(1.0202125612107882,
 20914,
 array([[  0,  49,   0,  51],
        [ 49, 103,  51, 105],
        [103, 150, 105, 150]]))

In [ ]:
start = time.time()
psdtw_prime_parallel_bsf_lb2(Q, C, r=r, l=l, P=P, dist_method=dist_method)
end = time.time()
print(f"Elapsed time: {end - start} seconds")

Elapsed time: 0.021781444549560547 seconds


# Nearest neighbor search with bsf

In [ ]:
r = 0.1
l = 2.0
P = 4
dist_method = 1
function_used = psdtw_prime_parallel_bsf_lb2

In [ ]:
# Warmup for numba
start = time.time()
nearest_neighbor_search(query_set[0], target_set, r=r, l=l,  P=P, dist_method=dist_method, dist_func=function_used)
end = time.time()
elapsed_time = end - start
print(elapsed_time)

3.300351619720459


In [ ]:
print("Starting nearest neighbor search over the entire query set...")
print(dataset_name)
all_count_dist_calls = []
start = time.time()
precision_at_1 = 0
for i in range(0, len(query_set)):
    best_idx, bsf, total_dist_calls = nearest_neighbor_search(query_set[i], target_set, r=r, l=l,  P=P, dist_method=dist_method, dist_func=function_used)
    all_count_dist_calls.append(total_dist_calls)
    precision_at_1 += 1 if best_idx == i else 0
print(
    f"{precision_at_1 / len(query_set):.2f}",
    end=" & ",
)
end = time.time()
elapsed_time = end - start
print()
print("Elapsed time: " + str(elapsed_time))
print("Average Elapsed time: " + str(elapsed_time / len(query_set)))

total_count_dist_calls = 0
for r in all_count_dist_calls:
    total_count_dist_calls += np.sum(r)
print("Total distance measure calls: " + str(total_count_dist_calls))

Starting nearest neighbor search over the entire query set...
GunPoint
0.64 & 
Elapsed time: 144.69158339500427
Average Elapsed time: 2.8938316679000855
Total distance measure calls: 386718825


In [ ]:
import datetime

print(f"This notebook was last run end-to-end on: {datetime.datetime.now()}\n")
###
###
###

This notebook was last run end-to-end on: 2025-12-10 22:55:36.222689

